# SGP4 against a two-body run

A sandbox for seeing what SGP4 does that a Keplerian orbit doesn't. Both start from the same
state: the SGP4 state at the element-set epoch, in GCRF (`cheap_prop.sgp4_epoch_states`, the
state the truth propagator starts from). Then one copy goes forward with SGP4 and the other with
a numerical integration of point-mass gravity, optionally plus J2.

The differences are expressed in rectilinear RTN axes of the SGP4 state (radial, along-track,
cross-track). The project's targets use the curvilinear version (`types.py`), but at these
sizes the picture is the same.

Nothing here writes to `src/` or to project data. Change the parameters in the cells marked
**Try it**.

In [ ]:
from pathlib import Path

import numpy as np
import pyarrow as pa
from scipy.integrate import solve_ivp

from orbit_correction import catalog
from orbit_correction import cheap_prop as C

REPO = Path.cwd().parent if Path.cwd().name == "practice" else Path.cwd()
elsets = catalog.load_celestrak_omm(REPO / "tests/fixtures/celestrak_gp_sample.json")

MU = 398600.8        # km^3/s^2, WGS-72 (SGP4's constants)
RE = 6378.135        # km, WGS-72
J2 = 0.001082616     # WGS-72

def pick(norad):
    i = elsets.column("norad_id").to_pylist().index(norad)
    return elsets.slice(i, 1)

iss = pick(25544)
row = iss.to_pylist()[0]
print(row["object_name"], row["epoch_utc"], "UTC")
print(f"perigee {row['perigee_alt_km']:.0f} km, apogee {row['apogee_alt_km']:.0f} km,",
      f"B* {row['bstar_per_earth_radii']:.3g} /ER")

ISS (ZARYA) 2026-11-22T21:08:45.053664 UTC
perigee 413 km, apogee 423 km, B* 0.000133 /ER


## The two propagators

`two_body` integrates point-mass gravity, plus J2 if asked. The J2 term takes the GCRF z axis as
Earth's pole, which ignores precession and nutation (tens of metres of pole offset over a week at
LEO radius): fine for a sandbox, not for the truth propagator.

In [ ]:
def two_body(r0, v0, t_s, j2=False):
    """Integrate r'' = -mu r / |r|^3 (+ J2) from (r0, v0) to the times t_s (s after start)."""
    def f(_, y):
        r, v = y[:3], y[3:]
        rn = np.linalg.norm(r)
        a = -MU * r / rn**3
        if j2:
            z2 = (r[2] / rn) ** 2
            k = 1.5 * J2 * MU * RE**2 / rn**5
            a = a + k * r * np.array([5 * z2 - 1, 5 * z2 - 1, 5 * z2 - 3])
        return np.concatenate([v, a])
    sol = solve_ivp(f, (t_s[0], t_s[-1]), np.concatenate([r0, v0]), t_eval=t_s,
                    method="DOP853", rtol=1e-12, atol=1e-9)
    return sol.y[:3].T, sol.y[3:].T

def rtn(r_ref, v_ref, r):
    """(R, T, N) of r_ref - r (reference minus other), rectilinear RTN of the reference, km."""
    ur = r_ref / np.linalg.norm(r_ref, axis=1, keepdims=True)
    h = np.cross(r_ref, v_ref)
    un = h / np.linalg.norm(h, axis=1, keepdims=True)
    ut = np.cross(un, ur)
    d = r_ref - r
    return np.stack([(d * ur).sum(1), (d * ut).sum(1), (d * un).sum(1)], axis=1)

dt = 60.0 * np.arange(7 * 1440 + 1)      # 7 days at 60 s
sgp4 = C.propagate_sgp4_from_epoch(iss, dt)[0].trajectory
start = C.sgp4_epoch_states(iss)[0].trajectory
print("epoch state (GCRF):", start.r_gcrf_km[0], "km")

epoch state (GCRF): [-3953.3369118  -5525.35137594    10.54428386] km


In [ ]:
HORIZONS_H = [1, 6, 24, 72, 168]

def report(label, r_other):
    d = rtn(sgp4.r_gcrf_km, sgp4.v_gcrf_km_s, r_other)
    print(f"{label}: SGP4 minus run, km")
    print("  horizon      R         T         N     |total|")
    for hr in HORIZONS_H:
        j = int(hr * 60)
        total = np.linalg.norm(d[j])
        print(f"  {hr:5d} h {d[j, 0]:9.3f} {d[j, 1]:9.3f} {d[j, 2]:9.3f} {total:9.3f}")

r_kep, _ = two_body(start.r_gcrf_km[0], start.v_gcrf_km_s[0], dt)
r_j2, _ = two_body(start.r_gcrf_km[0], start.v_gcrf_km_s[0], dt, j2=True)
report("two-body", r_kep)
report("two-body + J2", r_j2)

two-body: SGP4 minus run, km
  horizon      R         T         N     |total|
      1 h   -10.304    52.022    -7.991    53.631
      6 h     2.351   263.253    81.881   275.703
     24 h    82.414  1018.170  -456.994  1119.064
     72 h   808.740  2973.178 -1256.776  3327.662
    168 h  3926.134  5908.918 -1738.287  7304.210
two-body + J2: SGP4 minus run, km
  horizon      R         T         N     |total|
      1 h     0.042    -0.213     0.003     0.217
      6 h    -0.077    -1.112    -0.268     1.146
     24 h     0.377    -1.444     0.595     1.607
     72 h     1.095    14.496     1.580    14.623
    168 h     3.658   121.805     2.906   121.895


## What the numbers say

* **Two-body.** The difference is mostly along-track and cross-track and grows steadily. J2 makes
  the orbit plane precess (node regression, a few degrees per day in LEO) and moves the perigee
  and the mean motion; a Keplerian orbit has none of that, so the plane drifts apart
  (cross-track) and the two objects fall out of phase (along-track). Past a few hundred km the
  rectilinear components stop meaning much (the axes no longer follow the orbit); read the
  total there.
* **Two-body + J2.** Most of that goes. What is left grows along-track: drag (SGP4's B* term
  lowers the orbit, which then speeds up), the other zonal terms SGP4 models, and the fact that
  the epoch state is *osculating* while SGP4 propagates *mean* elements. A numerical run from an
  osculating state carries the short-period J2 oscillation in its semimajor axis as a slightly
  wrong mean motion, which shows up as along-track drift.
* **What this sandbox doesn't show** is SGP4's own error, which is what the project learns. That
  needs a truth propagator with a full force model and real space weather (`truth_prop.py`).
  SGP4 drifts from truth because its drag model is a fitted B* with a fixed static atmosphere,
  its gravity is a truncated analytic theory, and its mean elements are fits to past tracking.

## Try it: change the drag term

Scale B* and watch the SGP4 along-track position move relative to the two-body + J2 run, which
has no drag at all. Drag lowers the orbit, which then speeds up, so SGP4 pulls ahead (positive
T). The drag-driven part (each row minus the B* x 0 row) scales with B* and grows with the
square of time.

In [ ]:
for scale in (0.0, 1.0, 2.0, 10.0):
    rec = {**row, "bstar_per_earth_radii": row["bstar_per_earth_radii"] * scale}
    t = pa.Table.from_pylist([rec], schema=elsets.schema)
    s = C.propagate_sgp4_from_epoch(t, dt)[0]
    d = rtn(s.trajectory.r_gcrf_km, s.trajectory.v_gcrf_km_s, r_j2[: len(s.trajectory)])
    line = "  ".join(f"{int(h):3d} h: {d[int(h * 60), 1]:9.3f}" for h in (24, 72, 168)
                     if int(h * 60) < len(d))
    print(f"B* x {scale:4.1f}  along-track (km)  {line}   status: {s.status.message}")

B* x  0.0  along-track (km)   24 h:    -4.624   72 h:   -14.164  168 h:   -34.326   status: ok
B* x  1.0  along-track (km)   24 h:    -1.444   72 h:    14.496  168 h:   121.805   status: ok
B* x  2.0  along-track (km)   24 h:     1.737   72 h:    43.186  168 h:   278.247   status: ok
B* x 10.0  along-track (km)   24 h:    27.225   72 h:   273.719  168 h:  1530.951   status: ok


## Other things to try

* Swap the ISS for another object: `pick(26998)` (TIMED, ~600 km), `pick(5398)` (IRAS, ~900 km),
  or `pick(28902)` (CZ-4B debris) and compare how fast the drag-driven part grows with altitude.
* Set `j2=True` and add a J3 term to `two_body`: how much of the remaining cross-track
  difference does it explain?
* Start the two-body run from an SGP4 state one orbit later instead of at the epoch, and see
  whether the along-track drift rate changes (the osculating-versus-mean effect).